# Week 3 - Sinh dataset ecommerce quy mô lớn (1GB+)

In [2]:
import os
import csv
import random
from datetime import date, timedelta

## Cấu hình

In [3]:
TARGET_GB = 1.2
OUTPUT_PATH = os.path.expanduser("~/intern_DE/Week_3/data/raw/ecommerce_sales_large.csv")

## Tham số sinh dữ liệu (giữ đúng phân phối dataset gốc Week 1-2)

In [4]:
CATEGORIES = ["Beauty", "Clothing", "Electronics"]
REGIONS = ["North", "South", "East", "West"]
PAYMENT_METHODS = ["Wallet", "Card"]  # dataset gốc chỉ có 2 loại, xem README Week 2

CATEGORY_WEIGHTS = [0.30, 0.40, 0.30]
REGION_WEIGHTS = [0.28, 0.27, 0.23, 0.22]
PAYMENT_WEIGHTS = [0.55, 0.45]

DATE_START = date(2022, 1, 1)
DATE_END = date(2035, 12, 31)
DATE_RANGE_DAYS = (DATE_END - DATE_START).days

HEADER = [
    "order_id", "order_date", "customer_id", "product_category", "region",
    "quantity", "unit_price", "discount", "payment_method", "delivery_days",
    "customer_rating", "revenue",
]

CHUNK_ROWS = 500_000

## Hàm sinh 1 dòng dữ liệu

In [5]:
def generate_row(order_id: int) -> list:
    category = random.choices(CATEGORIES, weights=CATEGORY_WEIGHTS, k=1)[0]
    region = random.choices(REGIONS, weights=REGION_WEIGHTS, k=1)[0]
    payment_method = random.choices(PAYMENT_METHODS, weights=PAYMENT_WEIGHTS, k=1)[0]

    order_date = DATE_START + timedelta(days=random.randint(0, DATE_RANGE_DAYS))
    customer_id = random.randint(1000, 50000)

    quantity = random.randint(1, 10)
    unit_price = round(random.uniform(10, 600), 2)
    discount = round(random.uniform(0, 0.4), 2)
    delivery_days = random.randint(1, 14)
    customer_rating = round(random.uniform(1, 5), 1)
    revenue = round(quantity * unit_price * (1 - discount), 2)

    return [
        order_id, order_date.isoformat(), customer_id, category, region,
        quantity, unit_price, discount, payment_method, delivery_days,
        customer_rating, revenue,
    ]

## Sinh dữ liệu theo chunk cho đến khi đạt dung lượng mục tiêu

In [6]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
target_bytes = TARGET_GB * (1024 ** 3)

order_id = 1
with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(HEADER)

    while True:
        rows = [generate_row(order_id + i) for i in range(CHUNK_ROWS)]
        writer.writerows(rows)
        order_id += CHUNK_ROWS

        f.flush()
        current_size = os.path.getsize(OUTPUT_PATH)
        current_gb = current_size / (1024 ** 3)
        print(f"[PROGRESS] Đã sinh {order_id - 1:,} dòng | "
              f"Dung lượng hiện tại: {current_gb:.2f} GB / {TARGET_GB} GB")

        if current_size >= target_bytes:
            break

final_size_gb = os.path.getsize(OUTPUT_PATH) / (1024 ** 3)
print(f"\n[DONE] Hoàn tất: {order_id - 1:,} dòng, {final_size_gb:.2f} GB tại {OUTPUT_PATH}")

[PROGRESS] Đã sinh 500,000 dòng | Dung lượng hiện tại: 0.03 GB / 1.2 GB
[PROGRESS] Đã sinh 1,000,000 dòng | Dung lượng hiện tại: 0.07 GB / 1.2 GB
[PROGRESS] Đã sinh 1,500,000 dòng | Dung lượng hiện tại: 0.10 GB / 1.2 GB
[PROGRESS] Đã sinh 2,000,000 dòng | Dung lượng hiện tại: 0.14 GB / 1.2 GB
[PROGRESS] Đã sinh 2,500,000 dòng | Dung lượng hiện tại: 0.17 GB / 1.2 GB
[PROGRESS] Đã sinh 3,000,000 dòng | Dung lượng hiện tại: 0.21 GB / 1.2 GB
[PROGRESS] Đã sinh 3,500,000 dòng | Dung lượng hiện tại: 0.24 GB / 1.2 GB
[PROGRESS] Đã sinh 4,000,000 dòng | Dung lượng hiện tại: 0.28 GB / 1.2 GB
[PROGRESS] Đã sinh 4,500,000 dòng | Dung lượng hiện tại: 0.31 GB / 1.2 GB
[PROGRESS] Đã sinh 5,000,000 dòng | Dung lượng hiện tại: 0.34 GB / 1.2 GB
[PROGRESS] Đã sinh 5,500,000 dòng | Dung lượng hiện tại: 0.38 GB / 1.2 GB
[PROGRESS] Đã sinh 6,000,000 dòng | Dung lượng hiện tại: 0.41 GB / 1.2 GB
[PROGRESS] Đã sinh 6,500,000 dòng | Dung lượng hiện tại: 0.45 GB / 1.2 GB
[PROGRESS] Đã sinh 7,000,000 dòng | Dung

## Kiểm tra kết quả

In [8]:
import pandas as pd
preview = pd.read_csv(OUTPUT_PATH, nrows=5)
preview

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,1,2030-06-26,29918,Beauty,South,9,221.41,0.14,Wallet,5,4.6,1713.71
1,2,2034-05-25,28471,Clothing,North,4,554.94,0.02,Wallet,5,3.3,2175.36
2,3,2030-06-27,49419,Beauty,North,6,466.38,0.28,Wallet,5,2.9,2014.76
3,4,2022-01-07,28152,Electronics,East,5,191.37,0.30,Wallet,12,1.4,669.79
4,5,2023-03-25,13980,Beauty,South,10,165.45,0.16,Wallet,12,1.5,1389.78
